# CRONUS-A Data Analysis
The code below loads CRONUS-A data from a compilation spreadsheet and performs statistical analysis.

In [41]:
# Import packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd as tukeyhsd

In [42]:
# Basic Summary data of Jull et al. (2015)
jull = pd.DataFrame({'Mean':693000,'SD':44000, 'N':23, 'SE':44000/np.sqrt(23)}, index=['Jull'])

In [43]:
# Define a class lab to hold data for each lab or publication and process the data
class Pub:
    def __init__(self, name, data):
        self.name = name
        if not isinstance(data, np.ndarray):
            try:
                self.data = np.array(data)
            except Exception as e:
                raise ValueError("Data must be convertible to a numpy array") from e
        else:
            self.data = data 
    def __repr__(self):
        return f"Pub: {self.name}"
    # Append stats
    def append_stats(self):
        num_measurements = len(self.data)
        
        if num_measurements > 1:
            mean_val = np.mean(self.data[:, 0])
            std_er1 = np.sqrt(np.sum(self.data[:, 1]**2))/num_measurements # Chapter 3 in Bevington and Robinson (3.8-3.10 generally)
            std_er2 = np.std(self.data[:, 0], ddof=1)/np.sqrt(num_measurements) #1.9 and 4.14 in Bevington and Robinson
            std_er = max(std_er1, std_er2)

            std_dv1 = np.sqrt(np.sum(self.data[:, 1]**2)/num_measurements) # Chapter 3 in Bevington and Robinson (3.8-3.10 generally)
            std_dv2 = np.std(self.data[:, 0], ddof=1) #1.9 in Bevington and Robinson
            std_dv = max(std_dv1, std_dv2)
            if std_dv1 > std_dv2:
                print(f"Warning: For {self.name}, std_dv1 ({std_dv1}) is greater than std_dv2 ({std_dv2}).")
        else:
            if self.name == 'Pub 3M':
                num_measurements = 6 # Set n for Cologne to six data points
            mean_val = self.data[:, 0][0]
            std_er = self.data[:, 1][0]/np.sqrt(num_measurements)
            std_dv = self.data[:, 1][0]

        groupweight = 1/std_er**2
        return np.asarray([mean_val, std_er, groupweight, num_measurements, std_dv])

In [44]:
# Function list 
def initialize(filename, sheet=None, rows_to_skip=0, cols_to_use=None, flag_col='flag'):
    # Function to load data and separate by publication
    try:
        data = pd.read_excel(filename, skiprows=rows_to_skip, usecols=cols_to_use, sheet_name=sheet)
        classes = data[flag_col].unique()
        pubs = {}
        for cls in classes:
            pubs[f'pub{cls}'] = Pub(name=f'Pub {cls}', data=data[data[flag_col] == cls][['conc', 'conc_unc']].values)
        return pubs, data
    except Exception as e:
        raise ValueError("Error loading data from file") from e
def calculate_stats(pubs):
    # Function to load and process the data using the functions defined above
    df = []
    for pub_name, pub in pubs.items():
        stats = pub.append_stats()
        df.append({'Publication': pub_name, 'Mean': stats[0], 'Std Dev': stats[4], 'Weights': stats[2], 'n': stats[3]})
    summary_data = pd.DataFrame(df)
    return summary_data
def calculate_weighted_mean(data):
    # Calculate the weighted mean and uncertainty of the full dataset
    N = np.sum(data['n'].values)
    x = data['Mean'].values
    w = data['Weights'].values
    sum_weights = np.sum(w)
    weighted_mean = np.sum(w*x)/sum_weights #4.17 in Bevington and Robinson
    avg_std_mu = np.sqrt((((np.sum(w*(x**2))/np.sum(w))-weighted_mean**2)*(N/(N-1)))/N) #4.22 and 4.23 in Bevington and Robinson
    avg_std_samp = np.sqrt((((np.sum(w*(x**2))/np.sum(w))-weighted_mean**2)*(N/(N-1)))) #4.22 and 4.23 in Bevington and Robinson
    std_err = np.sqrt(1/sum_weights) # 4.19 in Bevington and Robinson
    unc = max(std_err, avg_std_mu)
    unc_type = 'Standard Error' if unc == std_err else 'Average Deviation of the Mean'
    return weighted_mean, unc, unc_type, N, avg_std_samp
def run_full_analysis(filename, sheet=None, rows_to_skip=0, cols_to_use=None, flag_col='flag'):
    # Combine all the functions above to run the full analysis and return the data
    pubs, data = initialize(filename, sheet, rows_to_skip, cols_to_use, flag_col)
    summary_data = calculate_stats(pubs)
    print(summary_data)
    weighted_mean, uncertainty, unc_type, N, measure_unc = calculate_weighted_mean(summary_data)
    print(f"Weighted Mean: {weighted_mean}, Uncertainty: {uncertainty}, Uncertainty Type: {unc_type}, Sample Uncertainty: {measure_unc}")
    summary_stats = {'Weighted Mean': weighted_mean, 'Uncertainty': uncertainty, 'N': N}
    return pubs, data, summary_stats, summary_data

In [45]:
# Load and process data
pubs, data, summary_stats, summary_data = run_full_analysis('PR_Data.xlsx', sheet='S3 - CRONUS-A Processing', rows_to_skip=2, cols_to_use='A:I')

  Publication           Mean       Std Dev       Weights     n
0        pub1  652263.333333  32734.260136  5.599461e-09   6.0
1        pub2  688516.666667  15420.711181  2.523146e-08   6.0
2       pub3M  672000.000000  71000.000000  1.190240e-09   6.0
3        pub4  709410.901438  38971.459147  8.559532e-09  13.0
4        pub5  727071.428571   7220.110802  1.342797e-07   7.0
5        pub6  662844.000000  67502.856718  3.291902e-09  15.0
6        pub7  705600.000000  27724.582954  1.170880e-08   9.0
7        pub8  601279.688979  50642.772118  6.628480e-09  17.0
8        pub9  708040.000000  16660.505909  2.882130e-08   8.0
Weighted Mean: 711743.5133227728, Uncertainty: 2841.8475262247366, Uncertainty Type: Average Deviation of the Mean, Sample Uncertainty: 26506.989088180875


In [46]:
# Calculate statistics for the Jull et al., 2015 paper using the same methods we apply to the intercomparison data
int_pubs, int_data, int_summary_stats, int_sumamry_data = run_full_analysis('PR_Data.xlsx', sheet='S4 - Intercomparison', rows_to_skip=1, cols_to_use='A:C')

  Publication           Mean       Std Dev       Weights    n
0        pub1  725000.000000  35870.136140  5.440415e-09  7.0
1        pub2  651666.666667  32690.467520  5.614473e-09  6.0
2        pub3  693375.000000  40031.906917  4.992033e-09  8.0
3        pub4  690000.000000   5000.000000  8.000000e-08  2.0
Weighted Mean: 689917.1359033668, Uncertainty: 3226.697690959476, Uncertainty Type: Standard Error, Sample Uncertainty: 12765.397011751393


In [47]:
# Comparison of our updated compilation mean to the Jull et al. (2015) consensus value
# Need to compare same statistics to each other so first calculate the SE of the compilation from the data and then combine with the SE of Jull et al. (2015) to get the combined SE for the z-test
se_compilation = np.std(data['conc'].values, ddof=1)/np.sqrt(len(data['conc'].values))
se_combined = np.sqrt(se_compilation**2 + jull['SE'].values[0]**2)
z_stat = (summary_stats['Weighted Mean'] - jull['Mean'].values[0]) / se_combined
p_value = 2 * (1 - stats.norm.cdf(np.abs(z_stat)))


if p_value < 0.05:
    print(f'The difference between the updated compilation mean and Jull et al. (2015) is statistically significant (p < 0.05).')
    print(f'z-statistic = {z_stat}, p-value = {p_value}')
else:
    print(f'The difference between the updated compilation mean and Jull et al. (2015) is not statistically significant (p >= 0.05).')
    print(f'z-statistic = {z_stat}, p-value = {p_value}')

The difference between the updated compilation mean and Jull et al. (2015) is not statistically significant (p >= 0.05).
z-statistic = 1.6605362599462898, p-value = 0.09680661966062898


In [48]:
# Determine if the means of each publication are significantly different from each other using a one-way ANOVA
# ANOVA for equal variances
anova_data = [pubs[pub].data[:, 0] for pub in pubs.keys()]
results = stats.f_oneway(*anova_data, equal_var=True)
print(f"ANOVA results: F-statistic = {results.statistic}, p-value = {results.pvalue}")
print()

# Statistical test for unequal variances (Welch's ANOVA)
anova_data = [pubs[pub].data[:, 0] for pub in pubs.keys() if len(pubs[pub].data) > 1]  # Only include publications with full datasets
results = stats.f_oneway(*anova_data, equal_var=False)
print(f"Welch's ANOVA results: F-statistic = {results.statistic}, p-value = {results.pvalue}")
print()

# Tukey HSD test for multiple comparisons
rng = np.random.default_rng(0)
data_for_tukey = []
labels_for_tukey = []

# Extract the mean and uncertinaty from the earlier calculations and generate artificial data for each publication based on the mean and uncertainty. This is needed because the Cologne data is reported without the raw data, so otherwise it will assume only a single data point which is incorrect. 
for _, row in summary_data.iterrows():
    pub, mean, sd, n = row['Publication'], row['Mean'], row['Std Dev'], int(row['n'])
    x = rng.standard_normal(n)
    x = (x - x.mean()) / x.std(ddof=1)   # z-scores
    x = mean + sd * x                     # artificial data from mean and uncertainty
    data_for_tukey.extend(x)
    labels_for_tukey.extend([pub] * n)

tukey_results = tukeyhsd(np.array(data_for_tukey), np.array(labels_for_tukey))
print(tukey_results)

ANOVA results: F-statistic = 9.991861257795465, p-value = 2.763619866395704e-09

Welch's ANOVA results: F-statistic = 28.731427133684196, p-value = 2.497782212239824e-10

       Multiple Comparison of Means - Tukey HSD, FWER=0.05       
group1 group2   meandiff   p-adj     lower        upper    reject
-----------------------------------------------------------------
  pub1   pub2   36253.3333 0.9046  -48008.1503 120514.8169  False
  pub1  pub3M   19736.6667 0.9979  -64524.8169 103998.1503  False
  pub1   pub4   57147.5681 0.2336  -14883.3961 129178.5323  False
  pub1   pub5   74808.0952 0.0951   -6388.3021 156004.4925  False
  pub1   pub6   10580.6667 0.9999  -59917.5485  81078.8818  False
  pub1   pub7   53336.6667 0.4083  -23583.1922 130256.5255  False
  pub1   pub8  -50983.6444 0.3278 -120286.8879  18319.5992  False
  pub1   pub9   55776.6667 0.3802   -23042.734 134596.0673  False
  pub2  pub3M  -16516.6667 0.9994 -100778.1503  67744.8169  False
  pub2   pub4   20894.2348 0.9908  -5

In [49]:
#Store summary stats for CRONUS-N data
CN_ETH1 = pd.DataFrame({'Mean': 12700, 'SD': 7000, 'N': 5, 'SE': 7000/np.sqrt(5)}, index=['CN_ETH1'])
CN_ETH2 = pd.DataFrame({'Mean': 12400, 'SD': 1700, 'N': 4, 'SE': 1700/np.sqrt(4)}, index=['CN_ETH2'])
CN_ANSTO = pd.DataFrame({'Mean': 32600, 'SD': 15900, 'N': 4, 'SE': 15900/np.sqrt(4)}, index=['CN_ANSTO'])

# Store Raw Data
CN_data = [[10857, 8101.8, 5130.4, 22375, 17175], [11200, 14200, 10600, 13500], [52178, 32207, 32775, 13144]]

# Welch's ANOVA (for unequal variances)
anova_results_CN = stats.f_oneway(*CN_data, equal_var=False)

# Print ANOVA results
print(f"ANOVA results for CN labs: F-statistic = {anova_results_CN.statistic}, p-value = {anova_results_CN.pvalue}")

ANOVA results for CN labs: F-statistic = 2.79562371785868, p-value = 0.15481114589076939


In [50]:
print(np.mean(CN_data[0]), np.std(CN_data[0], ddof=1)/np.sqrt(len(CN_data[0])))

12727.84 3126.230246415001
